<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 19 · Phân cụm</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Tìm nhóm trong dữ liệu không nhãn: k-means tính tay, khởi tạo, chọn số cụm bằng khuỷu tay và silhouette, rồi DBSCAN cho cụm hình dạng bất kỳ.</p>
</div>

### Sau notebook này bạn sẽ

- Tính tay k-means và hiểu vì sao nó luôn dừng.
- Thấy k-means kẹt ở đáy cục bộ và cách k-means++ chữa.
- Chọn số cụm bằng inertia và silhouette.
- Dùng DBSCAN và phân cụm phân cấp cho cụm không tròn.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch19-phan-cum/01-kmeans-tinh-tay/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · K-means tính tay](#trang-1)
2. [Trang 2 · Khởi tạo và scikit-learn](#trang-2)
3. [Trang 3 · Chọn số cụm](#trang-3)
4. [Trang 4 · DBSCAN và phân cụm phân cấp](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · K-means tính tay

K-means trên bảy điểm.

*Đọc kèm trên web: [Trang 1 · K-means tính tay](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch19-phan-cum/01-kmeans-tinh-tay/)*

In [ ]:
import numpy as np                                                 # tính toán
P = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5]])   # Bảng 19.1
tam = P[[0, 3]].copy()                                             # tâm khởi đầu: P1 và P4
for vong in range(1, 4):                                           # ba vòng
    d2 = ((P[:, None, :] - tam[None]) ** 2).sum(axis=2)            # bình phương khoảng cách, shape (7, 2)
    nhan = d2.argmin(axis=1)                                       # gán vào tâm gần nhất
    tam = np.array([P[nhan == k].mean(axis=0) for k in range(2)])  # dời tâm về trung bình
    qt = sum(((P[nhan == k] - tam[k]) ** 2).sum() for k in range(2))   # inertia với tâm mới
    print(vong, nhan, tam.round(4).tolist(), round(float(qt), 4))  # nhãn, tâm, inertia

Output mong đợi:

```text
1 [0 0 0 1 1 1 1] [[1.8333, 2.3333], [4.125, 5.375]] 12.2083
2 [0 0 1 1 1 1 1] [[1.25, 1.5], [3.9, 5.1]] 8.525
3 [0 0 1 1 1 1 1] [[1.25, 1.5], [3.9, 5.1]] 8.525
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">P3 hòa khoảng cách ở vòng 1 nên vào cụm 0, rồi đổi sang cụm 1 ở vòng 2 khi tâm cam trôi lại gần. Vòng 3 không ai đổi: hội tụ.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Khởi tạo và scikit-learn

Bốn đám điểm, khởi tạo ngẫu nhiên kẹt ở đáy cục bộ.

*Đọc kèm trên web: [Trang 2 · Khởi tạo và scikit-learn](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch19-phan-cum/02-khoi-tao-va-scikit-learn/)*

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering   # ba thuật toán
from sklearn.datasets import make_blobs, make_moons, load_iris       # ba bộ dữ liệu
from sklearn.metrics import adjusted_rand_score, silhouette_score, silhouette_samples   # thước đo
Xb, yb = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=0)   # 4 đám, giữ nhãn thật để chấm
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Xb)          # k-means++ với 10 lần chạy
print(round(km.inertia_, 1), round(adjusted_rand_score(yb, km.labels_), 4))   # inertia và ARI
qt = [KMeans(4, n_init=1, init="random", random_state=s).fit(Xb).inertia_ for s in range(10)]   # 10 lần khởi tạo ngẫu nhiên
print([round(v, 1) for v in qt])                                     # có lần kẹt

Output mong đợi:

```text
369.5 0.9463
[369.5, 369.5, 369.5, 369.5, 369.5, 669.3, 369.5, 369.5, 369.5, 669.3]
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI LÀ 669,3?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Ở hai lần kẹt, hai tâm chia nhau một đám, một tâm gộp hai đám khác. Bước gán và dời tâm không thoát ra được: một đáy cục bộ.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Chọn số cụm

Khuỷu tay, silhouette, và hoa iris.

*Đọc kèm trên web: [Trang 3 · Chọn số cụm](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch19-phan-cum/03-chon-so-cum/)*

In [ ]:
for k in range(1, 9):                                                  # k từ 1 tới 8
    kmk = KMeans(k, n_init=10, random_state=0).fit(Xb)                 # k-means
    print(k, round(kmk.inertia_, 1), round(silhouette_score(Xb, kmk.labels_), 4) if k > 1 else "-")   # inertia, silhouette

Output mong đợi:

```text
1 2978.6 -
2 1348.9 0.4964
3 708.8 0.5151
4 369.5 0.5801
5 326.3 0.5032
6 292.4 0.4305
7 260.2 0.3955
8 225.2 0.3444
```

In [ ]:
from sklearn.preprocessing import StandardScaler                    # chuẩn hóa
Xi, yi = load_iris(return_X_y=True)                                 # 150 hoa, 3 loài
Zi = StandardScaler().fit_transform(Xi)                             # chuẩn hóa trước khi đo khoảng cách
for k in [2, 3, 4]:                                                 # ba giá trị k
    li = KMeans(k, n_init=10, random_state=0).fit_predict(Zi)       # phân cụm
    print(k, round(silhouette_score(Zi, li), 4), round(adjusted_rand_score(yi, li), 4))   # silhouette và ARI so với loài thật

Output mong đợi:

```text
2 0.5818 0.5681
3 0.4599 0.6201
4 0.3869 0.4728
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Tin silhouette là sự thật.</b> Nó thích k = 2 vì setosa tách hẳn, hai loài kia chồng lấn. Số cụm đúng phụ thuộc câu hỏi bạn đặt ra.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · DBSCAN và phân cụm phân cấp

Hai vầng trăng: k-means, DBSCAN, phân cấp.

*Đọc kèm trên web: [Trang 4 · DBSCAN và phân cụm phân cấp](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch19-phan-cum/04-dbscan-va-phan-cap/)*

In [ ]:
Xm, ym = make_moons(300, noise=0.06, random_state=0)                       # hai vầng trăng ít nhiễu
km2 = KMeans(2, n_init=10, random_state=0).fit_predict(Xm)                 # k-means
db = DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm)                        # DBSCAN
ag = AgglomerativeClustering(2, linkage="single").fit_predict(Xm)          # phân cấp, single linkage
print(round(adjusted_rand_score(ym, km2), 4), round(adjusted_rand_score(ym, db), 4), int((db == -1).sum()), round(adjusted_rand_score(ym, ag), 4))   # ARI ba cách, số điểm nhiễu

Output mong đợi:

```text
0.2343 1.0 0 1.0
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Đổi linkage="single" thành "ward". ARI còn bằng 1 không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Tăng noise của make_moons lên 0,15. DBSCAN với eps = 0,2 còn tìm đúng hai vầng trăng không?</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch19-phan-cum/05-bai-tap/)*

### Bài 19.1 · Silhouette của P3

Tính tay $a$, $b$ và silhouette của P3 với cách chia cuối cùng.
<details><summary>Xem đáp án</summary>

$a \approx 1{,}808$, $b \approx 3{,}053$, $s \approx 0{,}408$, thấp nhất trong 7 điểm. Code ở ô ngay dưới.
</details>

In [ ]:
print(silhouette_samples(P, np.array([0, 0, 1, 1, 1, 1, 1])).round(4))   # silhouette từng điểm

Output mong đợi:

```text
[0.7777 0.7156 0.4076 0.5838 0.6925 0.687  0.6508]
```

### Bài 19.2 · k-means++ chọn tâm thứ hai

Với tâm đầu là P1, tính xác suất mỗi điểm được chọn làm tâm thứ hai.
<details><summary>Xem đáp án</summary>

Tỷ lệ với bình phương khoảng cách: P4 được 38%, P2 chỉ 0,9%. Code ở ô ngay dưới.
</details>

In [ ]:
d2 = ((P - P[0]) ** 2).sum(axis=1)       # bình phương khoảng cách tới P1
print(d2, (d2 / d2.sum()).round(4))      # và xác suất được chọn

Output mong đợi:

```text
[ 0.    1.25 13.   52.   22.25 28.25 18.5 ] [0.     0.0092 0.0961 0.3845 0.1645 0.2089 0.1368]
```

### Bài 19.3 · Chọn eps

Chạy DBSCAN với eps 0,05; 0,2; 1,0 trên hai vầng trăng, đếm số cụm và số điểm nhiễu.
<details><summary>Xem đáp án</summary>

0,05: 10 cụm vụn, 243 nhiễu. 0,2: đúng 2 cụm. 1,0: gộp thành 1 cụm. Code ở ô ngay dưới.
</details>

In [ ]:
for e in [0.05, 0.2, 1.0]:                                  # ba bán kính
    l = DBSCAN(eps=e, min_samples=5).fit_predict(Xm)        # DBSCAN
    print(e, len(set(l) - {-1}), int((l == -1).sum()))      # số cụm, số điểm nhiễu

Output mong đợi:

```text
0.05 10 243
0.2 2 0
1.0 1 0
```